In [51]:
%pip install langchain faiss-cpu sentence-transformers transformers accelerate
%pip install -U langchain langchain-community
%pip install pypdf
%pip install hf_xet
%pip install -U langchain-huggingface

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [198]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

import re
from sentence_transformers import CrossEncoder

### 1. Load PDf

In [199]:
# Load PDF
loader = PyPDFLoader("./Docs/France.pdf")  # Replace with your PDF path
docs = loader.load()
docs = docs[:31]

### 1.5 Pre-Process Text

In [200]:
def clean_text(text):
    # Replace newlines followed by lowercase letter with space (joining broken sentences)
    text = re.sub(r'\n(?=[a-z])', ' ', text)
    # Replace multiple spaces/newlines with single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


for doc in docs:
    doc.page_content = clean_text(doc.page_content)


### 2. Chunk Text with Overlap

In [201]:
# Chunk text with overlap (default chunk size ~1000 chars, overlap ~200)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(chunks)}")

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ---")
    print(chunk.page_content[:500])  # print first 500 chars to keep it manageable
    print()


Total chunks created: 188
--- Chunk 0 ---
French Republic République française Flag Coat of arms[I] Motto: "Liberté, égalité, fraternité" Anthem: "La Marseillaise" Diplomatic emblem France France,[IX] officially the French Republic,[X] is a country located primarily in Western Europe. Its overseas regions and territories include French Guiana in South America, Saint Pierre and Miquelon in the North Atlantic, the French West Indies, and many islands in Oceania and the Indian Ocean, giving it one of the largest discontiguous exclusive eco

--- Chunk 1 ---
Germany to the northeast; Switzerland to the east; Italy and Monaco to the southeast; Andorra and Spain to the south; and a maritime border with the United Kingdom to the northwest. Its metropolitan area extends from the Rhine to the Atlantic Ocean and from the Mediterranean Sea to the English Channel and the North Sea. Its eighteen integral regions—five of which are overseas—span a combined area of 632,702 km2 (244,288 sq mi) and have 

### 3. Create Embeddings Locally with Sentence Transformers

In [210]:
# Create embeddings locally with SentenceTransformers
embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")

### 4. Build FAISS index

In [211]:
# Build FAISS vector store from chunks
chunks = [doc for doc in chunks if not doc.page_content.strip().startswith(tuple("0123456789")) and len(doc.page_content) > 100]
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


### 5. Set up Local LLM 

In [212]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

tokenizer.model_max_length = 512
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_new_tokens=200)
llm = HuggingFacePipeline(pipeline=pipe)

Device set to use cpu


### 6. Create RetrievalQA Chain

In [217]:
prompt = PromptTemplate.from_template(
    "Answer the following question based only on the context provided.\n\nContext:\n{context}\n\nQuestion: {question}"
)


def get_context_text(docs, max_tokens=800):
    texts = [doc.page_content for doc in docs]
    tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    joined = ""
    total_tokens = 0
    for text in texts:
        tokens = tokenizer.encode(text, truncation=False, add_special_tokens=False)
        if total_tokens + len(tokens) > max_tokens:
            break
        joined += text + "\n\n"
        total_tokens += len(tokens)
    return joined.strip()


reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(question, docs):
    pairs = [(question, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)
    scored_docs = list(zip(docs, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, score in scored_docs[:5]]



def ask(question, show_chunks=True):
    docs_with_scores = vectorstore.search(question, search_type="similarity", k=5)

    docs = [doc for doc, score in docs_with_scores]     # docs_with_scores is List[Tuple[Document, float]]
    scores = [score for doc, score in docs_with_scores]


    docs_reranked = rerank(question, docs)
    context = get_context_text(docs_reranked)


    prompt_str = prompt.format(context=context, question=question)
    if show_chunks:
        print("\n--- Retrieved Chunks ---")
        for i, doc in enumerate(docs_reranked):
            print(f"\n[Chunk {i+1}]\n{doc.page_content[:500]}...")  # show first 500 chars
            print(f"\n[Chunk {i+1}] Score: {doc.metadata.get('score', 'N/A')}\n{doc.page_content[:400]}")
        print("\n------------------------")

    return llm.invoke(prompt_str)


### 7. Ask Questions

In [ ]:
print(llm.invoke(prompt_str))


ValueError: too many values to unpack (expected 2)